# Generating Templates

In [ ]:
import uuid
import pandas as pd
import json
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama

# ==========================================
# Step 1: Define the Strict Output Schema
# ==========================================
class MessageTemplate(BaseModel):
    hook_used: str = Field(description="The psychological hook or angle used in the message.")
    feature_reference: str = Field(description="The specific app feature referenced (e.g., Daily Streak, Leaderboard).")
    content_english: str = Field(description="The English push notification content.")
    content_hindi: str = Field(description="The precise Hindi translation of the push notification.")

class TemplateResponse(BaseModel):
    templates: List[MessageTemplate] = Field(description="A list containing exactly 5 distinct message templates.")

# ==========================================
# Step 2: Configuration & Model Init
# ==========================================
# Update these paths to match your project structure
PATH_SEGMENT_GOALS = './output/segment_lifecycle_goals.json'
PATH_SEGMENT_MAPPING = './output/segment_mapping.csv'
PATH_OUTPUT_CSV = './output/message_templates.csv'

# Initialize local Ollama model and bind the schema
# Change 'llama3' to whichever local model you are serving
base_llm = ChatOllama(model="llama3.2:latest", temperature=0.2)
structured_llm = base_llm.with_structured_output(TemplateResponse)

# ==========================================
# Step 3: Data Preparation (The Cross-Join)
# ==========================================
def prepare_tasks():
    with open(PATH_SEGMENT_GOALS, 'r') as f:
        json_data = json.load(f)
    
    df_json = pd.DataFrame(json_data['combinations'])[['segment_id', 'lifecycle_stage', 'primary_goal']]
    
    df_csv = pd.read_csv(PATH_SEGMENT_MAPPING)
    df_csv.rename(columns={
        'Segment ID': 'segment_id', 
        'Primary Theme (Octalysis)': 'primary_theme', 
        'Secondary Theme (Octalysis)': 'secondary_theme'
    }, inplace=True)
        
    df_merged = pd.merge(df_json, df_csv, on='segment_id')
    
    df_primary = df_merged.copy()
    df_primary['theme'] = df_primary['primary_theme']
    
    df_secondary = df_merged.copy()
    df_secondary['theme'] = df_secondary['secondary_theme']
    
    df_tasks = pd.concat([df_primary, df_secondary], ignore_index=True)
    df_tasks.drop(columns=['primary_theme', 'secondary_theme', 'Segment Name'], inplace=True, errors='ignore')
    
    return df_tasks.to_dict(orient='records')

# ==========================================
# Step 4: LLM Generator & Validation
# ==========================================
def generate_templates_for_row(llm_instance, row):
    system_prompt = "You are an expert copywriter for an English learning app. Our North Star metric is Daily Active Speaking Minutes."
    
    user_prompt = f"""Write exactly 5 push notification templates for a user in the {row['lifecycle_stage']} stage. 
Their current goal is: {row['primary_goal']}. 
You must use the Octalysis core drive of: {row['theme']}. 
"""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]

    try:
        # The invocation now returns a verified Pydantic object based on TemplateResponse
        result = llm_instance.invoke(messages)
        
        # Validation: Ensure it returned exactly 5 items
        if not result or len(result.templates) != 5:
            print(f"Validation failed for {row['segment_id']} - {row['theme']}: Returned {len(result.templates) if result else 0} items instead of 5.")
            return []
        
        # Extract and Inject Metadata
        enriched_templates = []
        for tpl in result.templates:
            enriched_templates.append({
                "template_id": f"TPL-{uuid.uuid4().hex[:8].upper()}",
                "segment_id": row['segment_id'],
                "lifecycle_stage": row['lifecycle_stage'],
                "goal": row['primary_goal'],
                "theme": row['theme'],
                "hook_used": tpl.hook_used,
                "feature_reference": tpl.feature_reference,
                "content_english": tpl.content_english,
                "content_hindi": tpl.content_hindi
            })
            
        return enriched_templates

    except Exception as e:
        print(f"Generation/Parsing Error for {row['segment_id']} - {row['theme']}: {str(e)}")
        return []

# ==========================================
# Step 5: Execution & Export
# ==========================================
def main():
    tasks_data = prepare_tasks()
    print(f"Prepared {len(tasks_data)} distinct prompt tasks.")
    
    all_results = []
    
    for index, row in enumerate(tasks_data):
        print(f"Processing task {index + 1}/{len(tasks_data)}: {row['segment_id']} - {row['theme']}")
        result = generate_templates_for_row(structured_llm, row)
        all_results.extend(result)
        
    print(f"\nSuccessfully generated {len(all_results)} valid templates.")
    
    if all_results:
        df_final = pd.DataFrame(all_results)
        cols = ['template_id', 'segment_id', 'lifecycle_stage', 'goal', 'theme', 
                'hook_used', 'feature_reference', 'content_english', 'content_hindi']
        df_final = df_final[cols]
        df_final.to_csv(PATH_OUTPUT_CSV, index=False)
        print(f"Exported to {PATH_OUTPUT_CSV} successfully.")
    else:
        print("No templates generated.")

# Execute the pipeline
main()

# Timing window finder using GMM

In [5]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
import warnings

# Suppress warnings from GMM when clusters are tiny
warnings.filterwarnings('ignore')

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_USER_SEGMENTS_CSV = './output/users_segmented.csv'
PATH_OUTPUT_GMM = './output/users_segmented_GMM.csv'

# ==========================================
# 1. Standard Window Mapping
# ==========================================
def map_hour_to_window(hour: float) -> str:
    """Maps a 0-23 hour float/int to the 6 predefined time windows."""
    # Handle NaN values safely
    if pd.isna(hour):
        return 'afternoon' # Safe default
        
    hour = int(round(hour)) % 24
    if 6 <= hour < 9: return 'early_morning'
    elif 9 <= hour < 12: return 'mid_morning'
    elif 12 <= hour < 15: return 'afternoon'
    elif 15 <= hour < 18: return 'late_afternoon'
    elif 18 <= hour < 21: return 'evening'
    else: return 'night'

# ==========================================
# 2. ML Optimizer Class
# ==========================================
class DynamicWindowOptimizer:
    def __init__(self, max_windows: int = 3):
        self.max_windows = max_windows
        self.segment_window_assignments = {}

    def fit(self, csv_path: str = PATH_USER_SEGMENTS_CSV):
        """Runs the ML algorithm to determine the optimal number of windows per segment."""
        # 1. Load Data from CSV
        df = pd.read_csv(csv_path)
        
        # Ensure 'preferred_hour' exists, fallback if column is named differently in your CSV
        hour_col = 'preferred_hour' if 'preferred_hour' in df.columns else 'hour'
        
        # Drop rows where the hour is missing just for the ML training
        df_clean = df.dropna(subset=[hour_col])
        
        results = []
        
        # 2. Process each segment individually
        for segment_id, group in df_clean.groupby('segment_id'):
            # Reshape for sklearn: 2D array of [[hour], [hour], ...]
            X = group[hour_col].values.reshape(-1, 1)
            
            # Edge Case: Very few users in segment, default to 1 window
            if len(X) < 5:
                mean_hour = X.mean()
                results.append({
                    'segment_id': segment_id,
                    'num_windows_assigned': 1,
                    'optimal_windows': [map_hour_to_window(mean_hour)],
                    'cluster_centers': [round(mean_hour, 1)]
                })
                self.segment_window_assignments[segment_id] = [map_hour_to_window(mean_hour)]
                continue

            # 3. Fit GMMs for k=1, 2, and 3 to find the optimal number of clusters
            best_gmm = None
            lowest_bic = np.inf
            
            # Test 1, 2, and 3 distinct time clusters
            max_k_to_try = min(self.max_windows, len(np.unique(X))) 
            
            for k in range(1, max_k_to_try + 1):
                gmm = GaussianMixture(n_components=k, random_state=42)
                gmm.fit(X)
                bic_score = gmm.bic(X)
                
                # If this model fits the data better (lower BIC), save it
                if bic_score < lowest_bic:
                    lowest_bic = bic_score
                    best_gmm = gmm
            
            # 4. Extract the optimal hours (cluster centers) from the best model
            optimal_hours = best_gmm.means_.flatten()
            
            # Map those hours back to your 6 standard text windows
            optimal_windows = list(set([map_hour_to_window(h) for h in optimal_hours]))
            
            results.append({
                'segment_id': segment_id,
                'num_windows_assigned': len(optimal_windows),
                'optimal_windows': optimal_windows,
                'cluster_centers': [round(h, 1) for h in optimal_hours]
            })
            
            # Save to internal state
            self.segment_window_assignments[segment_id] = optimal_windows

        return pd.DataFrame(results)

# ==========================================
# 3. Execution & Export
# ==========================================
if __name__ == "__main__":
    ml_optimizer = DynamicWindowOptimizer(max_windows=3)
    
    try:
        print(f"Loading data from {PATH_USER_SEGMENTS_CSV}...")
        df_optimal = ml_optimizer.fit(PATH_USER_SEGMENTS_CSV)
        
        print("\nML-Assigned Optimal Time Windows per Segment:")
        print(df_optimal.to_string(index=False))
        
        # Export the final optimal windows to the output directory
        df_optimal.to_csv(PATH_OUTPUT_GMM, index=False)
        print(f"\nSuccessfully exported GMM mappings to: {PATH_OUTPUT_GMM}")
        
    except FileNotFoundError:
        print(f"Error: Could not find '{PATH_USER_SEGMENTS_CSV}'. Check the path.")
    except KeyError as e:
        print(f"Error: Column {e} not found in the CSV. Please verify column names.")

Loading data from ./output/users_segmented.csv...

ML-Assigned Optimal Time Windows per Segment:
segment_id  num_windows_assigned                        optimal_windows   cluster_centers
        S1                     3       [evening, night, late_afternoon] [20.0, 5.0, 17.0]
        S2                     3          [mid_morning, evening, night] [8.7, 19.0, 22.0]
        S3                     1                       [late_afternoon]            [16.0]
        S4                     2               [early_morning, evening] [18.0, 7.0, 19.0]
        S5                     3    [early_morning, afternoon, evening] [7.0, 20.2, 13.0]
        S6                     3 [mid_morning, evening, late_afternoon] [19.6, 9.3, 15.0]
        S7                     2                   [mid_morning, night]       [20.5, 9.1]
        S8                     1                                [night]            [21.0]
        S9                     1                              [evening]            [19.2]

Su

# Compute each user frequency


In [6]:
import pandas as pd
import numpy as np
import random

# ==========================================
# 1. Frequency Logic Definition
# ==========================================
def assign_daily_frequency(activeness_score: float) -> int:
    """
    Takes an activeness score (0.0 to 1.0) and returns a dynamic 
    daily notification limit based on defined thresholds.
    """
    # Handle edge cases (missing data)
    if pd.isna(activeness_score):
        return random.randint(3, 4) # Default to lowest tier to be safe
        
    if activeness_score > 0.7:
        return random.randint(7, 9)
    elif 0.4 <= activeness_score <= 0.7:
        return random.randint(5, 6)
    else:
        return random.randint(3, 4)

# ==========================================
# 2. Execution & Data Processing
# ==========================================
def run_frequency_optimizer(input_csv: str, output_csv: str):
    print(f"Loading user data from {input_csv}...")
    
    # Load your user data
    df_users = pd.read_csv(input_csv)
    
    # Safety check: Ensure the column exists. If not, mock it for testing.
    if 'activeness_score' not in df_users.columns:
        print("Warning: 'activeness_score' not found. Generating mock scores for demonstration.")
        # Generates a random float between 0.0 and 1.0 for each user
        df_users['activeness_score'] = np.random.uniform(0.0, 1.0, len(df_users))
        
    # Apply the frequency logic row-by-row
    df_users['daily_notification_limit'] = df_users['activeness_score'].apply(assign_daily_frequency)
    
    # Show a quick summary of the distribution
    print("\nFrequency Distribution Summary:")
    print(df_users['daily_notification_limit'].value_counts().sort_index())
    
    # Save the updated dataframe
    df_users.to_csv(output_csv, index=False)
    print(f"\nSuccessfully exported updated user limits to: {output_csv}")

# ==========================================
# Run the pipeline
# ==========================================
if __name__ == "__main__":
    PATH_INPUT = './output/users_segmented.csv'
    PATH_OUTPUT = './output/users_frequency_assigned.csv'
    
    run_frequency_optimizer(PATH_INPUT, PATH_OUTPUT)

Loading user data from ./output/users_segmented.csv...

Frequency Distribution Summary:
daily_notification_limit
3    11
4    10
5     7
6    14
7     9
8     6
9     4
Name: count, dtype: int64

Successfully exported updated user limits to: ./output/users_frequency_assigned.csv


# Each user notification generation